# Modul 13: Production Readiness — Vom Prototyp zum Alltags-System

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du einen Health-Check-Monitor, Kosten-Tracker, Uptime-Kalkulator und eine Capstone-Checklist.

🎯 **Lernziel:** Ein Agent-System produktionsreif machen mit Monitoring, Kosten-Kontrolle und Runbook.

---

## Production = Observability + Reliability + Cost Control

```
  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
  │  MONITORING   │  │   BACKUP     │  │   KOSTEN     │
  │              │  │              │  │              │
  │ Health-Check │  │ Config-VCS   │  │ Cost/Task    │
  │ Latenz (p95) │  │ State-Backup │  │ Budget-Alert │
  │ Error-Rate   │  │ Recovery     │  │ Model-Mix    │
  └──────────────┘  └──────────────┘  └──────────────┘
```

Google SRE (2016): *Monitoring & Alerting* — übertragbar auf Agent-Systeme.

In [ ]:
# Health-Check-Monitor

import random
from datetime import datetime, timedelta


class HealthCheck:
    """Prueft den Status einer Komponente."""
    def __init__(self, name, check_fn):
        self.name = name
        self.check_fn = check_fn
        self.history = []

    def run(self):
        try:
            result = self.check_fn()
            status = 'healthy' if result else 'unhealthy'
        except Exception as e:
            status = 'error'
            result = str(e)
        entry = {'time': datetime.now().isoformat(), 'status': status, 'detail': result}
        self.history.append(entry)
        return entry


class HealthMonitor:
    """Ueberwacht alle Komponenten."""
    def __init__(self):
        self.checks = []

    def add(self, check):
        self.checks.append(check)

    def run_all(self):
        print('\n🏥 Health-Check Report')
        print(f'  Zeit: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
        print(f'  {"-"*50}')
        all_healthy = True
        for check in self.checks:
            result = check.run()
            icon = {'healthy': '🟢', 'unhealthy': '🟡', 'error': '🔴'}[result['status']]
            print(f'  {icon} {check.name:<20} {result["status"]:<12} {result["detail"]}')
            if result['status'] != 'healthy':
                all_healthy = False
        status = '✅ ALL SYSTEMS GO' if all_healthy else '⚠️ ISSUES DETECTED'
        print(f'  {"-"*50}')
        print(f'  {status}')
        return all_healthy


# Health-Checks definieren
monitor = HealthMonitor()

monitor.add(HealthCheck('Gateway', lambda: True))  # Immer OK
monitor.add(HealthCheck('LLM-API', lambda: random.random() > 0.1))  # 90% Uptime
monitor.add(HealthCheck('Memory-Store', lambda: True))
monitor.add(HealthCheck('Tool-Registry', lambda: True))
monitor.add(HealthCheck('Cron-Service', lambda: random.random() > 0.2))  # 80% Uptime

# Mehrere Checks simulieren
for i in range(3):
    monitor.run_all()
    print()

In [ ]:
# Kosten-Tracker + Uptime-Kalkulator

class CostTracker:
    """Verfolgt API-Kosten pro Task und Modell."""

    MODEL_PRICING = {
        'haiku':  {'input': 0.00025, 'output': 0.00125},  # pro 1K Tokens
        'sonnet': {'input': 0.003,   'output': 0.015},
        'opus':   {'input': 0.015,   'output': 0.075},
    }

    def __init__(self, daily_budget=1.0):
        self.daily_budget = daily_budget
        self.transactions = []

    def log_usage(self, model, input_tokens, output_tokens, task=''):
        pricing = self.MODEL_PRICING.get(model, self.MODEL_PRICING['sonnet'])
        cost = (input_tokens / 1000 * pricing['input']) + (output_tokens / 1000 * pricing['output'])
        entry = {
            'time': datetime.now().isoformat(),
            'model': model,
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'cost': cost,
            'task': task,
        }
        self.transactions.append(entry)
        return cost

    def daily_total(self):
        today = datetime.now().strftime('%Y-%m-%d')
        return sum(t['cost'] for t in self.transactions if t['time'].startswith(today))

    def budget_alert(self):
        spent = self.daily_total()
        pct = spent / self.daily_budget * 100
        if pct >= 90:
            return f'🔴 ALERT: {pct:.0f}% des Tagesbudgets verbraucht (${spent:.4f}/${self.daily_budget})'
        elif pct >= 70:
            return f'🟡 WARNUNG: {pct:.0f}% des Tagesbudgets (${spent:.4f}/${self.daily_budget})'
        else:
            return f'🟢 OK: {pct:.0f}% des Tagesbudgets (${spent:.4f}/${self.daily_budget})'

    def report(self):
        print('\n💰 Kosten-Report')
        print(f'  {"Task":<25} {"Modell":<8} {"In-Tok":<8} {"Out-Tok":<8} {"Kosten"}')
        print(f'  {"-"*65}')
        for t in self.transactions[-10:]:
            print(f'  {t["task"]:<25} {t["model"]:<8} {t["input_tokens"]:<8} {t["output_tokens"]:<8} ${t["cost"]:.4f}')
        print(f'  {"-"*65}')
        total = sum(t['cost'] for t in self.transactions)
        print(f'  GESAMT: ${total:.4f}')
        print(f'  {self.budget_alert()}')


class UptimeCalculator:
    """Berechnet Uptime aus Health-Check-History."""

    @staticmethod
    def calculate(checks, period_hours=24):
        print(f'\n⏱️ Uptime-Report (letzte {period_hours}h)')
        for check in checks:
            total = len(check.history)
            healthy = sum(1 for h in check.history if h['status'] == 'healthy')
            uptime = healthy / max(total, 1) * 100
            bar_len = int(uptime / 5)
            bar = '█' * bar_len + '░' * (20 - bar_len)
            print(f'  {check.name:<20} {bar} {uptime:.1f}% ({healthy}/{total})')


# === Demo ===
tracker = CostTracker(daily_budget=1.00)

# Typische Nutzung simulieren
tracker.log_usage('haiku', 500, 200, 'Health-Check')
tracker.log_usage('haiku', 800, 300, 'Daily Digest')
tracker.log_usage('sonnet', 2000, 1500, 'Deep Research')
tracker.log_usage('sonnet', 1500, 800, 'Content Creation')
tracker.log_usage('haiku', 300, 100, 'Memory Recall')
tracker.log_usage('opus', 3000, 2000, 'Complex Analysis')

tracker.report()

# Uptime berechnen
UptimeCalculator.calculate(monitor.checks)

In [ ]:
# 🎯 ÜBUNG: Erstelle ein Runbook + Capstone-Checklist
#
# Aufgabe:
# 1. Definiere ein Runbook als Dictionary mit 3 Playbooks:
#    - 'api_down': Was tun wenn die API nicht erreichbar ist?
#    - 'budget_exceeded': Was tun wenn das Budget ueberschritten ist?
#    - 'injection_detected': Was tun bei Prompt-Injection?
# 2. Jedes Playbook hat: 'severity', 'steps' (Liste), 'escalate_to'
# 3. Erstelle eine Capstone-Checklist die alle Module abfragt

runbook = {
    'api_down': {
        'severity': '',       # TODO: 'HIGH', 'MEDIUM', 'LOW'
        'steps': [],          # TODO: 3-5 Schritte als Strings
        'escalate_to': '',    # TODO: Wer wird benachrichtigt?
    },
    # TODO: 'budget_exceeded' und 'injection_detected'
}

# TODO: Capstone-Checklist
# checklist = {
#     'M1': {'topic': 'Was ist ein Agent?', 'check': 'Kann 5 Komponenten benennen', 'done': False},
#     ...
# }

In [ ]:
# ✅ LÖSUNG

runbook = {
    'api_down': {
        'severity': 'HIGH',
        'steps': [
            '1. Status-Page des API-Anbieters pruefen',
            '2. Letzte Logs auf Fehler pruefen (Error 429, 500, 503)',
            '3. Fallback-Modell aktivieren (z.B. Haiku statt Sonnet)',
            '4. Automatische Tasks pausieren (Cron stoppen)',
            '5. User informieren: "API voruebergehend nicht verfuegbar"',
        ],
        'escalate_to': 'admin (Telegram)',
    },
    'budget_exceeded': {
        'severity': 'MEDIUM',
        'steps': [
            '1. Kosten-Report pruefen: Welcher Task war teuer?',
            '2. Auf guenstigeres Modell wechseln (Opus → Sonnet → Haiku)',
            '3. Cron-Intervalle vergroessern',
            '4. Tagesbudget ggf. anpassen',
        ],
        'escalate_to': 'admin (Dashboard-Alert)',
    },
    'injection_detected': {
        'severity': 'HIGH',
        'steps': [
            '1. Verdaechtige Nachricht loggen (Audit-Log)',
            '2. Aktion NICHT ausfuehren',
            '3. User-Session einfrieren',
            '4. Admin benachrichtigen mit Details',
            '5. Pattern zum Injection-Detektor hinzufuegen',
        ],
        'escalate_to': 'admin (sofort, Telegram)',
    },
}

print('=== Runbook ===\n')
for name, playbook in runbook.items():
    print(f'📕 {name} (Severity: {playbook["severity"]})')
    for step in playbook['steps']:
        print(f'   {step}')
    print(f'   → Eskalation: {playbook["escalate_to"]}')
    print()

# Capstone-Checklist
print('\n=== 🎓 Capstone-Checklist ===')
print('Pruefe ob du alle Module verstanden hast:\n')

checklist = [
    ('M1',  'Was ist ein Agent?',       '5 Komponenten + Autonomie-Spektrum'),
    ('M2',  'Agent Loop',               'ReAct-Loop + Reflection erklaeren'),
    ('M3',  'Architektur',              '4 Schichten zuordnen koennen'),
    ('M4',  'Tool Use',                 'Read-only vs Side-Effects + Policies'),
    ('M5',  'Persona',                  'Konsistenz-Test ≥80%'),
    ('M6',  'Gedaechtnis',              '3-stufiges Memory implementiert'),
    ('M7a', 'Skills nutzen',            'Skill installiert + Routing getestet'),
    ('M7b', 'Skills bauen',             'SKILL.md + Script + Tests'),
    ('M8',  'Debugging',                '5 Fehlermodi + Ethik-Checkliste'),
    ('M9',  'Multi-Agent',              '2-Agent-System mit Message-Passing'),
    ('M10', 'Automatisierung',          'Cron + Event-Trigger + Kosten'),
    ('M11', 'Security',                 'Injection-Detektor + Blocklist + Audit'),
    ('M12', 'Orchestrierung',           'Task-Dekomposition + Aggregation'),
    ('M13', 'Production',               'Health-Monitor + Kosten + Runbook'),
]

for modul, topic, check in checklist:
    print(f'  [ ] {modul:<5} {topic:<25} → {check}')

print(f'\n📊 {len(checklist)} Module. Ziel: Alle Checks bestanden!')
print('\n🎉 Gratulation! Du hast den Agent Systems Kurs abgeschlossen.')

## Production-Readiness-Matrix

| Dimension | Frage | Tool |
|-----------|-------|------|
| Monitoring | Weiss ich wenn etwas kaputt ist? | Health-Check |
| Kosten | Weiss ich was es kostet? | Cost-Tracker |
| Backup | Kann ich wiederherstellen? | Config-VCS |
| Security | Bin ich vor Angriffen geschuetzt? | Audit-Log |
| Runbook | Weiss ich was zu tun ist bei Problemen? | Playbooks |

---

### ✅ Self-Check
- [ ] Ich kann einen Health-Check-Monitor aufsetzen
- [ ] Ich kann API-Kosten tracken und Budgets setzen
- [ ] Ich habe ein Runbook mit mindestens 3 Playbooks
- [ ] Ich habe die Capstone-Checklist durchgearbeitet

**🎓 Kurs abgeschlossen! → Weiter zur [Community & Showcase](https://github.com/janrummel/agent-systems)**